In [2]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import os
from io import BytesIO

import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

BUCKET_NAME = os.environ["MINIO_BUCKET"]

s3_client = boto3.client(
    "s3",
    endpoint_url=os.environ["MINIO_ENDPOINT"],
    aws_access_key_id=os.environ["MINIO_ACCESS_KEY"],
    aws_secret_access_key=os.environ["MINIO_SECRET_KEY"],
    region_name="us-east-1",
    config=Config(signature_version="s3v4"),
)

In [4]:
def read_parquet_prefix(s3_client, bucket_name, prefix):
    """Read every Parquet object under a MinIO prefix."""
    paginator = s3_client.get_paginator("list_objects_v2")
    parquet_keys = []

    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            if key.endswith(".parquet"):
                parquet_keys.append(key)

    if not parquet_keys:
        raise FileNotFoundError(
            f"No Parquet files found under prefix: {prefix}"
        )

    dataframes = []
    for key in parquet_keys:
        response = s3_client.get_object(Bucket=bucket_name, Key=key)
        dataframe = pd.read_parquet(
            BytesIO(response["Body"].read()),
            engine="fastparquet",
        )
        dataframes.append(dataframe)
        print(f"Loaded: {key}")
        print(f"Shape: {dataframe.shape}")

    return pd.concat(dataframes, ignore_index=True)


def read_parquet_object(s3_client, bucket_name, object_key):
    response = s3_client.get_object(Bucket=bucket_name, Key=object_key)
    return pd.read_parquet(
        BytesIO(response["Body"].read()),
        engine="fastparquet",
    )

In [5]:
metadata_df = read_parquet_prefix(
    s3_client,
    BUCKET_NAME,
    "metadata/etl_metadata/"
)

required_gold_datasets = {"train_features", "test_features"}
successful_gold_metadata = metadata_df[
    (metadata_df["layer"] == "gold")
    & (metadata_df["status"] == "success")
    & metadata_df["dataset_name"].isin(required_gold_datasets)
]

datasets_by_run = successful_gold_metadata.groupby("run_id")["dataset_name"].apply(set)
valid_gold_run_ids = sorted(
    run_id
    for run_id, datasets in datasets_by_run.items()
    if required_gold_datasets.issubset(datasets)
)

if not valid_gold_run_ids:
    raise FileNotFoundError(
        "No successful Gold run contains train_features and test_features data."
    )

pipeline_run_id = valid_gold_run_ids[-1]
selected_gold_metadata = successful_gold_metadata[
    successful_gold_metadata["run_id"] == pipeline_run_id
]
pipeline_load_date = selected_gold_metadata["load_date"].iloc[0]

def gold_object_key(dataset_name):
    matches = selected_gold_metadata[
        selected_gold_metadata["dataset_name"] == dataset_name
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Gold object for {dataset_name}.")
    return matches.iloc[0]["object_key"]

train_features = read_parquet_object(
    s3_client, BUCKET_NAME, gold_object_key("train_features")
)
test_features = read_parquet_object(
    s3_client, BUCKET_NAME, gold_object_key("test_features")
)

required_silver_datasets = {"RUL_data"}
successful_silver_metadata = metadata_df[
    (metadata_df["layer"] == "silver")
    & (metadata_df["status"] == "success")
    & metadata_df["dataset_name"].isin(required_silver_datasets)
]

selected_silver_metadata = successful_silver_metadata[
    successful_silver_metadata["run_id"] == pipeline_run_id
]
pipeline_load_date = selected_silver_metadata["load_date"].iloc[0]

def silver_object_key(dataset_name):
    matches = selected_silver_metadata[
        selected_silver_metadata["dataset_name"] == dataset_name
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one Silver object for {dataset_name}.")
    return matches.iloc[0]["object_key"]

rul_df_minio = read_parquet_object(
    s3_client, BUCKET_NAME, silver_object_key("RUL_data")
)

print(f"Selected Gold run: {pipeline_run_id}")


Loaded: metadata/etl_metadata/load_date=2026-09-02/run_id=20260902T120041719003Z/manifest.parquet
Shape: (9, 8)
Selected Gold run: 20260902T120041719003Z


In [6]:
rul_df_minio = rul_df_minio.drop(columns=["processing_timestamp","data_quality_status"])

In [7]:
test_features = test_features.merge(rul_df_minio,on="unit_number",how="left")

In [8]:
last_observed_cycle = test_features.groupby("unit_number")["time_in_cycles"].max().reset_index()

In [9]:
test_features = test_features.merge(last_observed_cycle,on="unit_number",how="left")

In [ ]:
test_features = test_features.rename(columns={"time_in_cycles_x":"time_in_cycles"})

In [13]:
test_features = test_features.rename(columns={"time_in_cycles_y":"last_observed_cycle"})

In [15]:
test_features["remaining_useful_life"] = test_features["remaining_useful_life"] + test_features["last_observed_cycle"] - test_features["time_in_cycles"]

In [17]:
test_features = test_features.drop(columns=["last_observed_cycle"])

In [19]:
if test_features["remaining_useful_life"].isna().any():
    raise ValueError(
        "Some test engines have no official RUL label."
    )

In [28]:
test_features.head()

,unit_number,time_in_cycles,operational_setting_1,operational_setting_2,operational_setting_3,current_sensor_measurement_1,current_sensor_measurement_2,current_sensor_measurement_3,current_sensor_measurement_4,current_sensor_measurement_5,current_sensor_measurement_6,current_sensor_measurement_7,current_sensor_measurement_8,current_sensor_measurement_9,current_sensor_measurement_10,current_sensor_measurement_11,current_sensor_measurement_12,current_sensor_measurement_13,current_sensor_measurement_14,current_sensor_measurement_15,current_sensor_measurement_16,current_sensor_measurement_17,current_sensor_measurement_18,current_sensor_measurement_19,current_sensor_measurement_20,current_sensor_measurement_21,sensor_measurement_1_mean_20,sensor_measurement_2_mean_20,sensor_measurement_3_mean_20,sensor_measurement_4_mean_20,sensor_measurement_5_mean_20,sensor_measurement_6_mean_20,sensor_measurement_7_mean_20,sensor_measurement_8_mean_20,sensor_measurement_9_mean_20,sensor_measurement_10_mean_20,sensor_measurement_11_mean_20,sensor_measurement_12_mean_20,sensor_measurement_13_mean_20,sensor_measurement_14_mean_20,sensor_measurement_15_mean_20,sensor_measurement_16_mean_20,sensor_measurement_17_mean_20,sensor_measurement_18_mean_20,sensor_measurement_19_mean_20,sensor_measurement_20_mean_20,sensor_measurement_21_mean_20,sensor_measurement_1_std_20,sensor_measurement_2_std_20,sensor_measurement_3_std_20,sensor_measurement_4_std_20,sensor_measurement_5_std_20,sensor_measurement_6_std_20,sensor_measurement_7_std_20,sensor_measurement_8_std_20,sensor_measurement_9_std_20,sensor_measurement_10_std_20,sensor_measurement_11_std_20,sensor_measurement_12_std_20,sensor_measurement_13_std_20,sensor_measurement_14_std_20,sensor_measurement_15_std_20,sensor_measurement_16_std_20,sensor_measurement_17_std_20,sensor_measurement_18_std_20,sensor_measurement_19_std_20,sensor_measurement_20_std_20,sensor_measurement_21_std_20,sensor_measurement_1_min_20,sensor_measurement_2_min_20,sensor_measurement_3_min_20,sensor_measurement_4_min_20,sensor_measurement_5_min_20,sensor_measurement_6_min_20,sensor_measurement_7_min_20,sensor_measurement_8_min_20,sensor_measurement_9_min_20,sensor_measurement_10_min_20,sensor_measurement_11_min_20,sensor_measurement_12_min_20,sensor_measurement_13_min_20,sensor_measurement_14_min_20,sensor_measurement_15_min_20,sensor_measurement_16_min_20,sensor_measurement_17_min_20,sensor_measurement_18_min_20,sensor_measurement_19_min_20,sensor_measurement_20_min_20,sensor_measurement_21_min_20,sensor_measurement_1_max_20,sensor_measurement_2_max_20,sensor_measurement_3_max_20,sensor_measurement_4_max_20,sensor_measurement_5_max_20,sensor_measurement_6_max_20,sensor_measurement_7_max_20,sensor_measurement_8_max_20,sensor_measurement_9_max_20,sensor_measurement_10_max_20,sensor_measurement_11_max_20,sensor_measurement_12_max_20,sensor_measurement_13_max_20,sensor_measurement_14_max_20,sensor_measurement_15_max_20,sensor_measurement_16_max_20,sensor_measurement_17_max_20,sensor_measurement_18_max_20,sensor_measurement_19_max_20,sensor_measurement_20_max_20,sensor_measurement_21_max_20,sensor_measurement_1_difference,sensor_measurement_2_difference,sensor_measurement_3_difference,sensor_measurement_4_difference,sensor_measurement_5_difference,sensor_measurement_6_difference,sensor_measurement_7_difference,sensor_measurement_8_difference,sensor_measurement_9_difference,sensor_measurement_10_difference,sensor_measurement_11_difference,sensor_measurement_12_difference,sensor_measurement_13_difference,sensor_measurement_14_difference,sensor_measurement_15_difference,sensor_measurement_16_difference,sensor_measurement_17_difference,sensor_measurement_18_difference,sensor_measurement_19_difference,sensor_measurement_20_difference,sensor_measurement_21_difference,sensor_measurement_1_%change,sensor_measurement_2_%change,sensor_measurement_3_%change,sensor_measurement_4_%change,sensor_measurement_5_%change,sensor_measurement_6_%change,sensor

In [17]:
pd.set_option('display.max_columns', None)

In [20]:
def data_preprocessing(df):
    modeling_df = df[df["time_in_cycles"]>19].copy()
    return modeling_df


In [18]:
def shuffle_engines(df):
    engine_ids = [int(x) for x in df["unit_number"].unique()]

    rng = np.random.default_rng(seed=42)
    ids_shuffled = rng.permutation(engine_ids)
    return ids_shuffled

def data_split(df,ids_shuffled,start=0,end=1):
    validation_engine_ids = ids_shuffled[int(start*len(ids_shuffled)):int(end*len(ids_shuffled))]
    train_engine_ids = [x for x in ids_shuffled if x not in validation_engine_ids]

    training_rows = df[df["unit_number"].isin(train_engine_ids)]
    validation_rows = df[df["unit_number"].isin(validation_engine_ids)]

    if not set(train_engine_ids).isdisjoint(validation_engine_ids):
        raise ValueError("An engine exists in both training and validation data")

    X_train = training_rows.drop(columns=["unit_number","remaining_useful_life"])
    y_train = training_rows["remaining_useful_life"]

    X_validation = validation_rows.drop(columns=["unit_number","remaining_useful_life"])
    y_validation = validation_rows["remaining_useful_life"]

    constant_columns = [
        column for column in X_train.columns
        if X_train[column].nunique(dropna=False) == 1
    ]
    X_train = X_train.drop(columns=constant_columns)
    X_validation = X_validation.drop(columns=constant_columns)

    return [
        X_train,
        y_train,
        X_validation,
        y_validation,
        validation_rows,
        train_engine_ids,
        validation_engine_ids,
        constant_columns,
    ]


In [21]:
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score

def evaluation(model_name,y_validation,predictions,validation_rows):
    model_predictions = pd.Series(predictions,index=y_validation.index)

    model_mae = mean_absolute_error(y_validation,model_predictions)
    model_rmse = mean_squared_error(y_validation,model_predictions) ** 0.5
    model_r2 = r2_score(y_validation,model_predictions)

    model_comparison = pd.concat([y_validation,model_predictions],axis=1)
    model_comparison = model_comparison.rename(columns={0:"predicted_rul"})

    critical_rows = model_comparison[model_comparison["remaining_useful_life"]<=10]
    warning_rows = model_comparison[
        (model_comparison["remaining_useful_life"]>10)
        & (model_comparison["remaining_useful_life"]<=30)
    ]
    healthy_rows = model_comparison[model_comparison["remaining_useful_life"]>30]

    critical_mae = mean_absolute_error(critical_rows["remaining_useful_life"],critical_rows["predicted_rul"])
    warning_mae = mean_absolute_error(warning_rows["remaining_useful_life"],warning_rows["predicted_rul"])
    healthy_mae = mean_absolute_error(healthy_rows["remaining_useful_life"],healthy_rows["predicted_rul"])

    critical_rmse = mean_squared_error(critical_rows["remaining_useful_life"],critical_rows["predicted_rul"]) ** 0.5
    warning_rmse = mean_squared_error(warning_rows["remaining_useful_life"],warning_rows["predicted_rul"]) ** 0.5
    healthy_rmse = mean_squared_error(healthy_rows["remaining_useful_life"],healthy_rows["predicted_rul"]) ** 0.5

    critical_error_mean = (critical_rows["remaining_useful_life"]-critical_rows["predicted_rul"]).mean()
    warning_error_mean = (warning_rows["remaining_useful_life"]-warning_rows["predicted_rul"]).mean()
    healthy_error_mean = (healthy_rows["remaining_useful_life"]-healthy_rows["predicted_rul"]).mean()

    predicted_validation_rows = validation_rows.copy()
    predicted_validation_rows["predicted_remaining_useful_life"] = model_predictions
    predicted_validation_rows["absolute_error"] = (
        predicted_validation_rows["remaining_useful_life"]
        - predicted_validation_rows["predicted_remaining_useful_life"]
    ).abs()
    predicted_validation_rows["squared_error"] = (
        predicted_validation_rows["remaining_useful_life"]
        - predicted_validation_rows["predicted_remaining_useful_life"]
    ) ** 2
    predicted_validation_rows["signed_error"] = (
        predicted_validation_rows["remaining_useful_life"]
        - predicted_validation_rows["predicted_remaining_useful_life"]
    )

    mae_by_engine = predicted_validation_rows.groupby("unit_number")["absolute_error"].mean()
    rmse_by_engine = np.sqrt(
        predicted_validation_rows.groupby("unit_number")["squared_error"].mean()
    )
    mean_error_by_engine = predicted_validation_rows.groupby("unit_number")["signed_error"].mean()

    predicted_validation_rows = predicted_validation_rows.sort_values(["unit_number","time_in_cycles"])
    prediction_change = (
        predicted_validation_rows
        .groupby("unit_number")["predicted_remaining_useful_life"]
        .diff()
        .dropna()
    )
    percentage_increasing = prediction_change.gt(0).mean()*100

    stability_by_engine = (
        predicted_validation_rows
        .groupby("unit_number")["predicted_remaining_useful_life"]
        .diff()
        .abs()
        .groupby(predicted_validation_rows["unit_number"])
        .mean()
    )

    missed_critical_observations = predicted_validation_rows[
        (predicted_validation_rows["remaining_useful_life"]<=10)
        & (predicted_validation_rows["predicted_remaining_useful_life"]>10)
    ]
    false_alarms = predicted_validation_rows[
        (predicted_validation_rows["remaining_useful_life"]>30)
        & (predicted_validation_rows["predicted_remaining_useful_life"]<=30)
    ]

    warning_cycles = predicted_validation_rows[
        (predicted_validation_rows["remaining_useful_life"]<=30)
        & (predicted_validation_rows["remaining_useful_life"]>10)
    ].copy()
    warning_cycles["received_warning"] = (
        warning_cycles["predicted_remaining_useful_life"]<=30
    ).astype(int)
    number_of_warning_per_engine = warning_cycles.groupby("unit_number")["received_warning"].sum()

    first_warning_rows = predicted_validation_rows[
        predicted_validation_rows["predicted_remaining_useful_life"]<=30
    ]
    first_warning = first_warning_rows.groupby("unit_number").first()

    engines_requiring_alert = (
        predicted_validation_rows
        .groupby("unit_number")["remaining_useful_life"]
        .min()
        .loc[lambda values: values <= 30]
        .index
    )

    engines_without_alert = set(
        engines_requiring_alert
    ) - set(first_warning.index)

    number_of_engines_without_alert = len(
        engines_without_alert
    )

    missed_critical_by_engine = missed_critical_observations["unit_number"].value_counts()
    engine_metrics = pd.DataFrame(index=mae_by_engine.index)
    engine_metrics.index.name = "unit_number"
    engine_metrics["mae"] = mae_by_engine
    engine_metrics["rmse"] = rmse_by_engine
    engine_metrics["signed_error"] = mean_error_by_engine
    engine_metrics["missed_critical_observations"] = missed_critical_by_engine.reindex(engine_metrics.index,fill_value=0)
    engine_metrics["warnings_per_engine"] = number_of_warning_per_engine.reindex(engine_metrics.index,fill_value=0)
    engine_metrics = engine_metrics.reset_index()

    successful_predicted_cycle = predicted_validation_rows[(predicted_validation_rows["remaining_useful_life"]<=30)& (predicted_validation_rows["predicted_remaining_useful_life"]<=30)]
    metrics = {
        "mae":float(model_mae),
        "rmse":float(model_rmse),
        "r2":float(model_r2),
        "critical_mae":float(critical_mae),
        "critical_rmse":float(critical_rmse),
        "critical_signed_error":float(critical_error_mean),
        "warning_mae":float(warning_mae),
        "warning_rmse":float(warning_rmse),
        "warning_signed_error":float(warning_error_mean),
        "healthy_mae":float(healthy_mae),
        "healthy_rmse":float(healthy_rmse),
        "healthy_signed_error":float(healthy_error_mean),
        "maximum_engine_mae":float(mae_by_engine.max()),
        "mean_prediction_change":float(prediction_change.mean()),
        "mean_absolute_prediction_change":float(prediction_change.abs().mean()),
        "percentage_predictions_increasing":float(percentage_increasing),
        "largest_prediction_jump":float(prediction_change.abs().max()),
        "number_of_missed_critical_observations":int(len(missed_critical_observations)),
        "missed_critical_percentage":float(
            len(missed_critical_observations)/len(critical_rows)*100
        ),
        "number_of_false_alarms":int(len(false_alarms)),
        "false_alarm_percentage":float(
            len(false_alarms)/len(healthy_rows)*100
        ),
        "failures_identified_early_percentage":float(
            number_of_warning_per_engine.gt(0).mean()*100
        ),
        "average_warning_lead_time":float(first_warning["remaining_useful_life"].mean()),
        "premature_alerts":int((first_warning["remaining_useful_life"]>30).sum()),
        "timely_alerts":int(
            (
                (first_warning["remaining_useful_life"]<=30)
                & (first_warning["remaining_useful_life"]>10)
            ).sum()
        ),
        "late_alerts":int((first_warning["remaining_useful_life"]<=10).sum()),
        "engines_without_alert":int(number_of_engines_without_alert),
        "total_operational_cost": int((400*int((first_warning["remaining_useful_life"]>30).sum())) + (1000*(int((first_warning["remaining_useful_life"]<=10).sum())+int(number_of_engines_without_alert))) + (300*int(first_warning.index.nunique()))),
    }

    return {
        "metrics":metrics,
        "engine_metrics":engine_metrics,
        "unstable_engine":stability_by_engine.loc[[stability_by_engine.idxmax()]],
        "predicted_validation_rows":predicted_validation_rows
    }

In [ ]:
import time
from pathlib import Path

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import optuna
import sklearn
from sklearn.ensemble import RandomForestRegressor

# Store MLflow runs locally inside this project.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
mlflow_database = (project_root / "mlflow.db").resolve().as_posix()
mlflow.set_tracking_uri(f"sqlite:///{mlflow_database}")
mlflow.set_experiment("cmapss_phase_6_random_forest_operational_cost_folds")

processed_train_features = data_preprocessing(train_features)
engine_ids = shuffle_engines(processed_train_features)

def objective(trial):
    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}") as child_run:
        params = {
            "n_estimators":trial.suggest_int("n_estimators",40,160),
            "max_features":None,
            "min_samples_leaf":trial.suggest_int("min_samples_leaf",1,50),
            "n_jobs":10,
            "random_state":42,
        }

        mlflow.log_params(params)
        mlflow.log_params({
            "model_type":"RandomForestRegressor",
            "window_size":20,
            "warning_threshold":30,
            "critical_threshold":10,
            "validation_fraction":0.2,
        })
        mlflow.log_params({
            "missed_failure_cost": 1000,
            "false_alarm_cost": 400,
            "maintenance_intervention_cost": 300,
        })
        mlflow.set_tags({
            "gold_run_id":str(pipeline_run_id),
            "dataset":"FD001",
            "purpose":"random_forest_tuning",
        })

        results = {}
        total_training_duration = 0
        for x in range(0,9,2):
            data = data_split(processed_train_features,engine_ids,start=x/10,end=(x+2)/10)
            X_train = data[0]
            y_train = data[1]
            X_validation = data[2]
            y_validation = data[3]
            validation_rows = data[4]
            train_engine_ids = data[5]
            validation_engine_ids = data[6]
            constant_columns = data[7]
            
            mlflow.log_dict({"features": list(X_train.columns),
                            "training_engine_ids": [int(x) for x in train_engine_ids],
                            "validation_engine_ids": [int(x)for x in validation_engine_ids],
                            "constant_columns_removed":constant_columns,},
                            f"metadata/fold_{x}.json",
                            )

            training_start = time.perf_counter()
            random_forest = RandomForestRegressor(**params)
            random_forest.fit(X_train,y_train)
            training_duration = time.perf_counter()-training_start

            validation_predictions = random_forest.predict(X_validation)
            results[f"fold_{x}"] = evaluation(
                "Random Forest",
                y_validation,
                validation_predictions,
                validation_rows,
            )

            results[f"fold_{x}"]["engine_metrics"]["fold"] = int(x/2) + 1

            total_training_duration  += training_duration

        averaged_metrics = {}
        for metric_name,value in results["fold_0"]["metrics"].items():
            averaged_metrics[f"{metric_name}_mean"] = np.mean([value,
                                                       results["fold_2"]["metrics"][metric_name], 
                                                       results["fold_4"]["metrics"][metric_name],
                                                       results["fold_6"]["metrics"][metric_name],
                                                       results["fold_8"]["metrics"][metric_name]])
            averaged_metrics[f"{metric_name}_std"] = np.std([value,
                                                       results["fold_2"]["metrics"][metric_name], 
                                                       results["fold_4"]["metrics"][metric_name],
                                                       results["fold_6"]["metrics"][metric_name],
                                                       results["fold_8"]["metrics"][metric_name]])

        mlflow.log_metric("training_duration_seconds",total_training_duration )
        mlflow.log_metrics(averaged_metrics)
        all_engine_metrics  = pd.concat([results["fold_0"]["engine_metrics"],
                                                results["fold_2"]["engine_metrics"],
                                                results["fold_4"]["engine_metrics"],
                                                results["fold_6"]["engine_metrics"],
                                                results["fold_8"]["engine_metrics"]],
                                                ignore_index=True,)
        
        mlflow.log_table(
            data=all_engine_metrics,
            artifact_file="tables/engine_metrics.json",
        )


        predicted_validation_rows  = pd.concat([results["fold_0"]["predicted_validation_rows"],
                                        results["fold_2"]["predicted_validation_rows"],
                                        results["fold_4"]["predicted_validation_rows"],
                                        results["fold_6"]["predicted_validation_rows"],
                                        results["fold_8"]["predicted_validation_rows"]],
                                        ignore_index=True,)

        predicted_validation_rows["absolute_error"] = (
                predicted_validation_rows["remaining_useful_life"]
                - predicted_validation_rows["predicted_remaining_useful_life"]
            ).abs()

        mae_by_engine = predicted_validation_rows.groupby("unit_number")["absolute_error"].mean()

        
        sorted_engine_mae = mae_by_engine.sort_values()
        unstable_engines = pd.concat([results["fold_0"]["unstable_engine"],
                                      results["fold_2"]["unstable_engine"],
                                      results["fold_4"]["unstable_engine"],
                                      results["fold_6"]["unstable_engine"],
                                      results["fold_8"]["unstable_engine"]])
        engines_to_plot = {
                            "worst_engine":int(sorted_engine_mae.index[-1]),
                            "best_engine":int(sorted_engine_mae.index[0]),
                            "average_engine":int(sorted_engine_mae.index[len(sorted_engine_mae)//2]),
                            "unstable_engine": int(unstable_engines.idxmax()),
                          }
        
        figures = {}
        for engine_name,engine_id in engines_to_plot.items():
            plot_data = predicted_validation_rows[
                predicted_validation_rows["unit_number"]==engine_id
            ]
    
            figure,axis = plt.subplots(figsize=(12,5))
            sns.lineplot(
                data=plot_data,
                x="time_in_cycles",
                y="remaining_useful_life",
                label="Actual RUL",
                ax=axis,
            )
            sns.lineplot(
                data=plot_data,
                x="time_in_cycles",
                y="predicted_remaining_useful_life",
                label="Predicted RUL",
                ax=axis,
            )
            axis.set_title(f"Random Forest prediction curves for {engine_name}")            
            axis.set_xlabel("Operating cycle")
            axis.set_ylabel("RUL")
            axis.grid(True)
            figures[f"{engine_name}"] = figure

        for figure_name,figure in figures.items():
            mlflow.log_figure(
                figure,
                f"plots/{figure_name}.png",
            )
            plt.close(figure)

        trial.set_user_attr("run_id",child_run.info.run_id)
        return averaged_metrics["total_operational_cost_mean"]

n_trials = 5
with mlflow.start_run(run_name="random_forest_operational_cost_study") as parent_run:
    mlflow.log_param("n_trials",n_trials)
    mlflow.log_param("number_of_folds",5)
    mlflow.set_tags({
        "gold_run_id":str(pipeline_run_id),
        "dataset":"FD001",
        "purpose":"optuna_study",
    })

    study = optuna.create_study(
        direction="minimize",
        study_name="random_forest_tuning",
    )
    study.optimize(objective,n_trials=n_trials)

    mlflow.log_params({
        f"best_{name}":value
        for name,value in study.best_trial.params.items()
    })
    mlflow.log_metric("best_operational_cost",study.best_value)

    best_run_id = study.best_trial.user_attrs.get("run_id")
    if best_run_id:
        mlflow.set_tag("best_child_run_id",best_run_id)

    best_parameters = study.best_trial.params
    best_operational_cost = study.best_value

print("Best parameters:", best_parameters)
print("Best mean operational cost:", best_operational_cost)
print("Best MLflow child run:",study.best_trial.user_attrs.get("run_id"))


2026/09/28 03:30:59 INFO mlflow.tracking.fluent: Experiment with name 'cmapss_phase_6_random_forest_operational_cost_folds' does not exist. Creating a new experiment.
[I 2026-09-28 03:30:59,483] A new study created in memory with name: random_forest_tuning
[I 2026-09-28 03:34:07,871] Trial 0 finished with value: 8560.0 and parameters: {'n_estimators': 153, 'min_samples_leaf': 2}. Best is trial 0 with value: 8560.0.
[I 2026-09-28 03:35:57,829] Trial 1 finished with value: 8640.0 and parameters: {'n_estimators': 140, 'min_samples_leaf': 34}. Best is trial 0 with value: 8560.0.
[I 2026-09-28 03:38:13,718] Trial 2 finished with value: 8720.0 and parameters: {'n_estimators': 106, 'min_samples_leaf': 1}. Best is trial 0 with value: 8560.0.
[I 2026-09-28 03:41:13,133] Trial 3 finished with value: 8560.0 and parameters: {'n_estimators': 154, 'min_samples_leaf': 2}. Best is trial 0 with value: 8560.0.
[I 2026-09-28 03:42:05,748] Trial 4 finished with value: 8560.0 and parameters: {'n_estimators

Best parameters: {'n_estimators': 153, 'min_samples_leaf': 2}
Best validation operational cost: 8560.0
Best MLflow child run: 9ec1ce8c389c40a5877e8c36a1778ea8


In [25]:
import time
from pathlib import Path

import mlflow
from mlflow.models import infer_signature
from sklearn.ensemble import RandomForestRegressor
from mlflow import MlflowClient

from pathlib import Path

import mlflow
from mlflow import MlflowClient

project_root = Path(
    r"C:\Users\Badr\Desktop\Python_3.0"
)

mlflow_database = (
    project_root / "mlflow.db"
).resolve().as_posix()

mlflow.set_tracking_uri(
    f"sqlite:///{mlflow_database}"
)

client = MlflowClient()

print(
    "MLflow tracking URI:",
    mlflow.get_tracking_uri(),
)

experiment = client.get_experiment_by_name(
    "cmapss_phase_6_random_forest_operational_cost_folds"
)

if experiment is None:
    raise ValueError(
        "The tuning experiment was not found in MLflow."
    )

tuning_runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string=(
        "tags.mlflow.runName = "
        "'random_forest_operational_cost_study' "
        "and attributes.status = 'FINISHED' "
        f"and tags.gold_run_id = '{pipeline_run_id}'"
    ),
    order_by=["attributes.start_time DESC"],
    max_results=1,
)

if not tuning_runs:
    raise ValueError(
        "No completed Optuna tuning run was found "
        f"for Gold run {pipeline_run_id}."
    )

selected_tuning_run = tuning_runs[0]

best_parameters = {
    "n_estimators": int(
        selected_tuning_run.data.params[
            "best_n_estimators"
        ]
    ),
    "min_samples_leaf": int(
        selected_tuning_run.data.params[
            "best_min_samples_leaf"
        ]
    ),
}

best_operational_cost = float(
    selected_tuning_run.data.metrics[
        "best_operational_cost"
    ]
)

best_child_run_id = (
    selected_tuning_run.data.tags[
        "best_child_run_id"
    ]
)

best_child_run = client.get_run(
    best_child_run_id
)

best_child_run_name = best_child_run.data.tags[
    "mlflow.runName"
]

best_optuna_trial_number = int(
    best_child_run_name.removeprefix("trial_")
)

print(
    "Tuning run ID:",
    selected_tuning_run.info.run_id,
)
print("Best parameters:", best_parameters)
print(
    "Best operational cost:",
    best_operational_cost,
)
print(
    "Best Optuna trial:",
    best_optuna_trial_number,
)

MLflow tracking URI: sqlite:///C:/Users/Badr/Desktop/Python_3.0/mlflow.db
Tuning run ID: b692bdfb2e5e43d7abcc1e3d243a4241
Best parameters: {'n_estimators': 153, 'min_samples_leaf': 2}
Best operational cost: 8560.0
Best Optuna trial: 0


In [27]:
from mlflow import MlflowClient

with mlflow.start_run(
    run_name="final_random_forest_training"
) as final_run:

    # Describe the purpose of this run
    mlflow.set_tags({
    "run_type": "final_training_and_test_evaluation",
    "model_family": "random_forest",
    "selection_method": "optuna_5_fold_engine_validation",
    "gold_run_id": str(pipeline_run_id),
    "dataset": "FD001",
    "purpose": "final_model_evaluation",
})

    # Record where the selected parameters came from
    mlflow.log_param(
        "best_optuna_trial_number",
        best_optuna_trial_number,
    )
    mlflow.log_metric(
        "best_cross_validation_operational_cost",
        best_operational_cost,
    )

    # Record the selected model settings
    mlflow.log_params({
    "final_n_estimators": best_parameters["n_estimators"],
    "final_min_samples_leaf":
        best_parameters["min_samples_leaf"],
    "final_max_features": None,
    "final_random_state": 42,
    "final_n_jobs": -1,
})

    mlflow.log_params({
        "model_type":"RandomForestRegressor",
        "window_size":20,
        "warning_threshold":30,
        "critical_threshold":10,
        "validation_fraction":0.2,
    })
    mlflow.log_params({
        "missed_failure_cost": 1000,
        "false_alarm_cost": 400,
        "maintenance_intervention_cost": 300,
    })

    # Prepare all training engines
    processed_train_features = data_preprocessing(
        train_features
    )

    processed_test_features = data_preprocessing(
        test_features
    )

    X_train = processed_train_features.drop(
        columns=[
            "unit_number",
            "remaining_useful_life",
        ]
    )
    y_train = processed_train_features[
        "remaining_useful_life"
    ]
    
    X_test = processed_test_features.drop(
        columns=[
            "unit_number",
            "remaining_useful_life",
        ]
    )
    y_test = processed_test_features[
        "remaining_useful_life"
    ]

    # Learn constant columns from the complete training set
    constant_columns = [
        column
        for column in X_train.columns
        if X_train[column].nunique(dropna=False) == 1
    ]

    X_train = X_train.drop(
        columns=constant_columns
    )

    X_test = X_test.drop(columns=constant_columns)

    missing_test_features = set(X_train.columns) - set(X_test.columns)

    if missing_test_features:
        raise ValueError(
            f"Test data is missing features: {missing_test_features}"
        )

    X_test = X_test.reindex(columns=X_train.columns)

    # Record the training data information
    mlflow.log_params({
        "number_of_training_engines":
            processed_train_features["unit_number"].nunique(),
        "number_of_training_rows": len(X_train),
        "number_of_features": X_train.shape[1],
        "window_size": 20,
    })

    mlflow.log_dict(
        {
            "feature_columns": X_train.columns.tolist(),
            "constant_columns_removed": constant_columns,
        },
        "metadata/final_training_features.json",
    )

    # Build the selected model
    final_model = RandomForestRegressor(
        **best_parameters,
        max_features=None,
        random_state=42,
        n_jobs=-1,
    )

    # Measure and record training duration
    training_start = time.perf_counter()

    final_model.fit(
        X_train,
        y_train,
    )

    training_duration = (
        time.perf_counter() - training_start
    )

    mlflow.log_metric(
        "training_duration_seconds",
        training_duration,
    )

    final_model_predictions = final_model.predict(X_test)

    results = evaluation("Random Forest",
                         y_test,
                         final_model_predictions,
                         processed_test_features)

    mlflow.log_metrics(results["metrics"])
    mlflow.log_table(
        data = results["engine_metrics"],
        artifact_file="tables/engine_metrics.json",
        )

    predicted_test_rows = results["predicted_validation_rows"]
    
    predicted_test_rows["absolute_error"] = (
            predicted_test_rows["remaining_useful_life"]
            - predicted_test_rows["predicted_remaining_useful_life"]
        ).abs()

    mae_by_engine = predicted_test_rows.groupby("unit_number")["absolute_error"].mean()
    sorted_engine_mae = mae_by_engine.sort_values()
    unstable_engine_id = int(results["unstable_engine"].idxmax())

    engines_to_plot = {
                        "worst_engine":int(sorted_engine_mae.index[-1]),
                        "best_engine":int(sorted_engine_mae.index[0]),
                        "average_engine":int(sorted_engine_mae.index[len(sorted_engine_mae)//2]),
                        "unstable_engine": unstable_engine_id,
                        }

    figures = {}
    for engine_name,engine_id in engines_to_plot.items():
        plot_data = predicted_test_rows[
            predicted_test_rows["unit_number"]==engine_id
        ]

        figure,axis = plt.subplots(figsize=(12,5))
        sns.lineplot(
            data=plot_data,
            x="time_in_cycles",
            y="remaining_useful_life",
            label="Actual RUL",
            ax=axis,
        )
        sns.lineplot(
            data=plot_data,
            x="time_in_cycles",
            y="predicted_remaining_useful_life",
            label="Predicted RUL",
            ax=axis,
        )
        axis.set_title(f"Random Forest prediction curves for {engine_name}")            
        axis.set_xlabel("Operating cycle")
        axis.set_ylabel("RUL")
        axis.grid(True)
        figures[f"{engine_name}"] = figure

    for figure_name,figure in figures.items():
        mlflow.log_figure(
            figure,
            f"plots/{figure_name}.png",
        )
        plt.close(figure)

    model_signature = infer_signature(X_test,final_model_predictions)

    registered_model_name = (
        "cmapss_fd001_rul_random_forest"
    )

    logged_model = mlflow.sklearn.log_model(
        sk_model=final_model,
        name="model",
        signature=model_signature,
        input_example=X_test.head(5),
        registered_model_name=registered_model_name,
    )

    client = MlflowClient()

    model_versions = client.search_model_versions(
        f"name='{registered_model_name}' "
        f"and run_id='{final_run.info.run_id}'"
    )

    if len(model_versions) != 1:
        raise ValueError(
            "Expected exactly one registered model version "
            "for the final run."
        )

    registered_version = model_versions[0].version

    client.set_registered_model_alias(
        name=registered_model_name,
        alias="candidate",
        version=registered_version,
    )

    mlflow.set_tags({
        "registered_model_name": registered_model_name,
        "registered_model_version": registered_version,
        "registered_model_alias": "candidate",
    })

    print(
        "Registered model:",
        registered_model_name,
        "version:",
        registered_version,
    )

    endpoint_results = (
    predicted_test_rows
    .sort_values(["unit_number", "time_in_cycles"])
    .groupby("unit_number")
    .tail(1)
    )

    if len(endpoint_results) != processed_test_features["unit_number"].nunique():
        raise ValueError(
            "Endpoint evaluation must contain one row per test engine."
        )

    endpoint_mae = mean_absolute_error(
        endpoint_results["remaining_useful_life"],
        endpoint_results["predicted_remaining_useful_life"],
    )

    endpoint_rmse = mean_squared_error(
        endpoint_results["remaining_useful_life"],
        endpoint_results["predicted_remaining_useful_life"],
    ) ** 0.5

    endpoint_r2 = r2_score(
        endpoint_results["remaining_useful_life"],
        endpoint_results["predicted_remaining_useful_life"],
    )

    mlflow.log_metrics({
        "official_test_endpoint_mae": float(endpoint_mae),
        "official_test_endpoint_rmse": float(endpoint_rmse),
        "official_test_endpoint_r2": float(endpoint_r2),
    })

    print("Final run ID:", final_run.info.run_id)


c:\Users\Badr\Desktop\Python_3.0\.venv\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


Successfully registered model 'cmapss_fd001_rul_random_forest'.
Created version '1' of model 'cmapss_fd001_rul_random_forest'.


Registered model: cmapss_fd001_rul_random_forest version: 1
Final run ID: 360c5414c7d841a38bcf9480521a7b59


In [28]:
import mlflow
from mlflow import MlflowClient

mlflow.set_tracking_uri(
    "sqlite:///C:/Users/Badr/Desktop/Python_3.0/mlflow.db"
)

client = MlflowClient()

source_run_id = "360c5414c7d841a38bcf9480521a7b59"

target_experiment = client.get_experiment_by_name(
    "cmapss_phase_6_random_forest_operational_cost_folds"
)

if target_experiment is None:
    raise ValueError("Target experiment was not found.")

source_run = client.get_run(source_run_id)

with mlflow.start_run(
    experiment_id=target_experiment.experiment_id,
    run_name="final_random_forest_training_reference",
) as copied_run:

    mlflow.log_params(
        source_run.data.params
    )

    mlflow.log_metrics(
        source_run.data.metrics
    )

    mlflow.set_tags({
        "run_type": "final_model_reference",
        "copied_from_run_id": source_run_id,
        "registered_model_name":
            "cmapss_fd001_rul_random_forest",
        "registered_model_version": "1",
        "purpose":
            "comparison_with_cross_validation_runs",
    })

print("Comparison run ID:", copied_run.info.run_id)

Comparison run ID: 08f9cb42ee224fcf8298ce2d82589c5f


import time
from pathlib import Path

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature
import optuna
import sklearn
from sklearn.ensemble import RandomForestRegressor

# Store MLflow runs locally inside this project.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
mlflow_database = (project_root / "mlflow.db").resolve().as_posix()
mlflow.set_tracking_uri(f"sqlite:///{mlflow_database}")
mlflow.set_experiment("cmapss_phase_6_random_forest_operational_cost")

processed_train_features = data_preprocessing(train_features)
engine_ids = shuffle_engines(processed_train_features)
data = data_split(processed_train_features,engine_ids,start=0.8)
X_train = data[0]
y_train = data[1]
X_validation = data[2]
y_validation = data[3]
validation_rows = data[4]
train_engine_ids = data[5]
validation_engine_ids = data[6]
constant_columns = data[7]

def objective(trial):
    with mlflow.start_run(nested=True,run_name=f"trial_{trial.number}") as child_run:
        params = {
            "n_estimators":trial.suggest_int("n_estimators",40,160),
            "max_features":None,
            "min_samples_leaf":trial.suggest_int("min_samples_leaf",1,50),
            "n_jobs":10,
            "random_state":42,
        }

        mlflow.log_params(params)
        mlflow.log_params({
            "model_type":"RandomForestRegressor",
            "window_size":20,
            "warning_threshold":30,
            "critical_threshold":10,
            "validation_fraction":0.2,
        })
        mlflow.log_params({
            "missed_failure_cost": 1000,
            "false_alarm_cost": 400,
            "maintenance_intervention_cost": 300,
        })
        mlflow.set_tags({
            "gold_run_id":str(pipeline_run_id),
            "dataset":"FD001",
            "purpose":"random_forest_tuning",
        })
        mlflow.log_dict(
            {"features":list(X_train.columns)},
            "metadata/features.json",
        )
        mlflow.log_dict(
            {
                "training_engine_ids":[int(x) for x in train_engine_ids],
                "validation_engine_ids":[int(x) for x in validation_engine_ids],
                "constant_columns_removed":constant_columns,
            },
            "metadata/data_split.json",
        )

        training_start = time.perf_counter()
        random_forest = RandomForestRegressor(**params)
        random_forest.fit(X_train,y_train)
        training_duration = time.perf_counter()-training_start

        validation_predictions = random_forest.predict(X_validation)
        results = evaluation(
            "Random Forest",
            y_validation,
            validation_predictions,
            validation_rows,
        )

        mlflow.log_metric("training_duration_seconds",training_duration)
        mlflow.log_metrics(results["metrics"])
        mlflow.log_table(
            data=results["engine_metrics"],
            artifact_file="tables/engine_metrics.json",
        )

        for figure_name,figure in results["figures"].items():
            mlflow.log_figure(
                figure,
                f"plots/{figure_name}.png",
            )
            plt.close(figure)

        model_signature = infer_signature(X_validation,validation_predictions)
        mlflow.sklearn.log_model(
            sk_model=random_forest,
            name="model",
            signature=model_signature,
            input_example=X_validation.head(5),
        )

        trial.set_user_attr("run_id",child_run.info.run_id)
        return results["metrics"]["total_operational_cost"]

n_trials = 5
with mlflow.start_run(run_name="random_forest_operational_cost_study") as parent_run:
    mlflow.log_param("n_trials",n_trials)
    mlflow.log_param("number_of_folds",1)
    mlflow.set_tags({
        "gold_run_id":str(pipeline_run_id),
        "dataset":"FD001",
        "purpose":"optuna_study",
    })

    study = optuna.create_study(
        direction="minimize",
        study_name="random_forest_tuning",
    )
    study.optimize(objective,n_trials=n_trials)

    mlflow.log_params({
        f"best_{name}":value
        for name,value in study.best_trial.params.items()
    })
    mlflow.log_metric("best_operational_cost",study.best_value)

    best_run_id = study.best_trial.user_attrs.get("run_id")
    if best_run_id:
        mlflow.set_tag("best_child_run_id",best_run_id)

print("Best parameters:",study.best_trial.params)
print("Best validation operational cost:",study.best_value)
print("Best MLflow child run:",study.best_trial.user_attrs.get("run_id"))
